In [ ]:
from scipy.optimize import minimize
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats.distributions import norm
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.base.datetools import dates_from_str
from statsmodels.tsa.api import VARMAX
import statsmodels.api as sm
import pandas as pd
from scipy.linalg import inv

Code below are helper functions from the ARMA Lab

In [3]:
def kalman(F, Q, H, time_series):
    # Get dimensions
    dim_states = F.shape[0]

    # Initialize variables
    # covs[i] = P_{i | i-1}
    covs = np.zeros((len(time_series), dim_states, dim_states))
    mus = np.zeros((len(time_series), dim_states))

    # Solve of for first mu and cov
    covs[0] = np.linalg.solve(np.eye(dim_states**2) - np.kron(F, F), 
            np.eye(dim_states**2) @ Q.flatten()).reshape((dim_states, dim_states))
    mus[0] = np.zeros((dim_states,))

    # Update Kalman Filter
    for i in range(1, len(time_series)):
        # Assume u, R = 0
        SkInv = np.linalg.solve(H @ covs[i-1] @ H.T, np.eye(H.shape[0]))
        Kk_H_Pkk = covs[i-1] @ (H.T @ (SkInv @ (H @ covs[i-1])))
        covs[i] = F @ ((covs[i-1] - Kk_H_Pkk) @ F.T) + Q
        mus[i] = F @ mus[i-1] + (F @ (covs[i-1] @ (H.T @ SkInv))) @ (time_series[i-1] - H @ mus[i-1])
    return mus, covs

def state_space_rep(phis, thetas, mu, sigma):
    # Initialize variables
    dim_states = max(len(phis), len(thetas)+1)
    dim_time_series = 1 # hardcoded for 1d time_series

    F = np.zeros((dim_states, dim_states))
    Q = np.zeros((dim_states, dim_states))
    H = np.zeros((dim_time_series, dim_states))

    # Create F
    F[0, :len(phis)] = phis
    F[1:, :-1] = np.eye(dim_states - 1)
    # Create Q
    Q[0][0] = sigma**2
    # Create H
    H[0][0] = 1.
    H[0][1:len(thetas)+1] = thetas

    return F, Q, H, dim_states, dim_time_series

In [ ]:
# loading in cleaned data
produce_data = pd.read_csv("cleaned_produce.csv")

# TODO make individual fruit data covariance stationary and 
# rememeber to standardize/normalize by sub mean and div by std

Grid searching for best p and q values in ARIMA model

In [ ]:
def sm_arma(data, p_max=3, q_max=3, n=30):
    """
    Build an ARMA model with statsmodel and
    predict future n values.

    Parameters:
        data: data prepped to be fed into ARMA model
        p_max (int): maximum order of autoregressive model
        q_max (int): maximum order of moving average model
        n (int): number of values to predict

    Return:
        aic (float): aic of optimal model
        best_p
        best_q
    """
    
    # initializing paramters to optimize
    lowestAIC = np.inf
    best_p = p_max
    best_q = q_max

    # grid search p and q
    for p in range(1, p_max + 1):
        for q in range(1, q_max + 1):

            # MAY NEED TO CHANGE THE TREND PARAMETER!!!!!!!!!!!!!!!
            model = ARIMA(data, order=(p, 0, q), trend="c")
            results = model.fit()

            aic = results.aic
            print(f"aic: {aic}, p: {p}, q: {q}")
            
            # tracking best aic and paramters
            if aic < lowestAIC:
                lowestAIC = aic
                best_p = p
                best_q = q
     
    return lowestAIC, best_p, best_q